# Arm 7' (joint training with FracAtlas), seeds 43 and 44

The pilot (seed 42) passed the pre-set advance rule (+1.57 pts mAP50-95, positive in 4/5 folds). This notebook adds seeds 43 and 44 on the 5 folds (10 runs, about 55 min each, estimated ~4.5-5 h on T4 x2) so that arm 7' has 3 seeds like the baseline. Same code and settings as notebook 05 (VARIANT generic, RUN_PRE off).
Settings: GPU T4 x2, Internet ON, **both datasets added**. The run may approach a session limit: if it is cut off, Save Version still keeps finished runs (each run writes its own metrics.json).

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/nithoukhan1/hbfmid.git"
REPO = "/kaggle/working/hbfmid"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.215"], check=True)
os.environ["WANDB_MODE"] = "disabled"
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2, then restart."
print("repo commit:", subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
DATA_ROOT = "/kaggle/input/datasets/nithoukhan2/hbfmid/Human Bone Fractures Multi-modal Image Dataset (HBFMID)/Bone Fractures Detection"
FA_ROOT = "/kaggle/input/datasets/nithoukhan/fracatlasfigshare/FracAtlas"
assert os.path.isdir(DATA_ROOT), DATA_ROOT
assert os.path.isdir(FA_ROOT + "/images"), FA_ROOT
SMOKE = False
VARIANT = "generic"
MODEL = "yolov8s.pt"
SEEDS = [43, 44]
WORK = "/kaggle/working/fa_data"
if SMOKE:
    EPOCHS, OUT, FOLDS, LIMIT = 2, "/kaggle/working/results/arm7_seeds_smoke", [0], 40
else:
    EPOCHS, OUT, FOLDS, LIMIT = 100, "/kaggle/working/results/arm7_seeds", [0, 1, 2, 3, 4], 0
d = f"{WORK}/fa_{VARIANT}_c10"
if not os.path.exists(d + "/summary.json"):
    subprocess.run([sys.executable, f"{REPO}/scripts/data/prepare_fracatlas_yolo.py", "--fa-root", FA_ROOT,
                    "--manifest", f"{REPO}/data_splits/fracatlas/manifest.csv", "--out", d, "--variant", VARIANT,
                    "--cls-id", "10", "--limit", str(LIMIT)], check=True)
ARM7 = f"fajoint_{VARIANT}"

def make_cmd(fold, seed):
    def f(gpu):
        return [sys.executable, f"{REPO}/scripts/training/run_baseline.py", "--mode", "clean", "--fold", str(fold), "--seed", str(seed),
                "--model", MODEL, "--epochs", str(EPOCHS), "--fixed-epochs", "--device", str(gpu), "--data-root", DATA_ROOT,
                "--splits-dir", f"{REPO}/data_splits", "--out", OUT, "--arm", ARM7,
                "--extra-train", f"{d}/lists/fractured_all.txt", "--extra-names", "FracAtlas fracture"]
    return f
TASKS = [(f"clean_f{f}_s{s}_{ARM7}", make_cmd(f, s), None) for s in SEEDS for f in FOLDS]
print(len(TASKS), "runs, epochs =", EPOCHS, "->", OUT)

In [ ]:
import queue, threading, time
os.makedirs(f"{OUT}/logs", exist_ok=True)
gpus = list(range(torch.cuda.device_count()))
q = queue.Queue()
for t in TASKS:
    q.put(t)
failed = []

def worker(gpu):
    while True:
        try:
            name, cmd_fn, needs = q.get_nowait()
        except queue.Empty:
            return
        while needs and not os.path.exists(needs):          # e.g. arm 6 waits for the FracAtlas pre-trained checkpoint
            if any(n.startswith("fa_pre_") for n in failed):
                break
            time.sleep(20)
        t0 = time.time()
        with open(f"{OUT}/logs/{name}.log", "w") as f:
            rc = subprocess.run(cmd_fn(gpu), stdout=f, stderr=subprocess.STDOUT).returncode
        print(f"[GPU {gpu}] {name}: {'OK' if rc == 0 else 'FAILED rc=' + str(rc)} in {(time.time()-t0)/60:.1f} min", flush=True)
        if rc != 0:
            failed.append(name)

threads = [threading.Thread(target=worker, args=(g,)) for g in gpus]
[t.start() for t in threads]; [t.join() for t in threads]
print("failed runs:", failed or "none")
for n in failed:
    print("\n==== tail of", n)
    print("".join(open(f"{OUT}/logs/{n}.log").readlines()[-25:]))

In [ ]:
import shutil
BASE = [f"{REPO}/results/controls", f"{REPO}/results/baseline_fixed"]
PREV = f"{REPO}/results/arms_fa_generic"          # seed-42 runs of arm 7' (pushed earlier)
subprocess.run([sys.executable, f"{REPO}/scripts/evaluation/aggregate_arm.py", "--name", f"arm_{ARM7}_s43_44", "--runs", OUT, "--arm-tag", ARM7, "--out", OUT])
subprocess.run([sys.executable, f"{REPO}/scripts/evaluation/compare_arms.py", "--data-root", DATA_ROOT, "--baseline-runs", *BASE,
                "--arm-runs", OUT, PREV, "--arm-name", ARM7, "--arm-tag", ARM7, "--out", f"{OUT}/{ARM7}_3seeds_vs_baseline"])
shutil.make_archive("/kaggle/working/arm7_seeds_results", "zip", OUT)
print(sorted(os.listdir(OUT)))
if failed:
    print("\nWARNING: failed runs:", failed)

After the run: download `arm7_seeds_results.zip`, unzip once into `results/arm7_seeds/`, commit and push.